# PharmaOpt — Notebook 12 | Strict CMS Model Readiness Audit

**Purpose:** Audit the corrected Notebook 11 output, compare it with the *legacy* v0.5 panel, and identify whether the historical FDA labels are suitable for model training. **Read-only**: no MySQL, no training, no CSV writes, no file deletions. Your original Notebook 12 remains unchanged.

**Important:** A three-year CMS lag is a conservative timing rule, not independent proof of CMS publication availability. The 2025 cohort has already been examined repeatedly, so it is a development holdout, not an untouched test set. FDA `shortage_target` is a specific retrospective NDC-linked event definition; it should not be described as every real-world drug shortage.

**How to run:** In JupyterLab, open this notebook and choose **Run → Run All Cells**. It works even if Notebook 11 has only completed the merge and has not yet exported the new panel.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path(r"C:\Users\jpiza\PharmaOpt")
OUT = ROOT / "outputs"
STRICT_DIR = OUT / "cms_strict_review_v01"

legacy_panel = OUT / "pharmaopt_integrated_package_year_v05.csv"
strict_panel = STRICT_DIR / "pharmaopt_integrated_package_year_v05.csv"
strict_v04_panel = STRICT_DIR / "pharmaopt_v04_model_panel_rebuilt.csv"
legacy_scores = OUT / "pharmaopt_2025_model_scores_v05.csv"
strict_scores = STRICT_DIR / "pharmaopt_2025_model_scores_v05.csv"
source_zip = ROOT / "drug-shortages-0001-of-0001.json.zip"

paths = {
    "Legacy v0.5 panel (old CMS)": legacy_panel,
    "Strict v0.4 rebuilt panel": strict_v04_panel,
    "Strict v0.5 integrated panel": strict_panel,
    "Legacy model scores": legacy_scores,
    "Strict model scores (optional)": strict_scores,
    "FDA shortage snapshot": source_zip,
}
print("PHARMAOPT — MODEL READINESS PREFLIGHT")
for label, path in paths.items():
    print(f"{'FOUND  ' if path.is_file() else 'NOT YET'} | {label}: {path}")
print("\nRead-only audit. Nothing was modified.")

## 1. Confirm which panel was actually exported

The corrected CMS merge passed in Notebook 11, but that does **not** by itself prove that a corrected integrated CSV was exported. This cell checks the corrected v0.4 and v0.5 panels separately, validates the package-year grain, and tests the CMS timing for matched rows.

In [ ]:
def audit_panel(path, label):
    if not path.is_file():
        print(f"{label}: NOT EXPORTED YET — no changes needed here.")
        return None
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    needed = ["package_ndc", "prediction_year", "shortage_target"]
    missing = sorted(set(needed) - set(cols))
    if missing:
        raise ValueError(f"{label}: missing essential columns: {missing}")
    optional = [c for c in ["cms_data_year", "cms_demand_match", "cms_total_claims"] if c in cols]
    frame = pd.read_csv(path, usecols=needed + optional,
                        dtype={"package_ndc": "string"}, low_memory=False)
    frame["prediction_year"] = pd.to_numeric(frame["prediction_year"], errors="raise").astype(int)
    print("\n" + "=" * 65)
    print(label)
    print("Rows:", f"{len(frame):,}")
    print("Duplicate package-year keys:", f"{frame.duplicated(['package_ndc','prediction_year']).sum():,}")
    print("Missing package NDCs:", f"{frame['package_ndc'].isna().sum():,}")
    print("Invalid labels:", f"{(~frame['shortage_target'].isin([0, 1])).sum():,}")
    display(frame.groupby('prediction_year')['shortage_target'].agg(['size', 'sum']))
    if 'cms_data_year' in frame:
        if 'cms_demand_match' in frame:
            matched = frame['cms_demand_match'].eq(1)
        else:
            matched = frame['cms_data_year'].notna()
        mismatched = (~(frame.loc[matched, 'prediction_year'] - frame.loc[matched, 'cms_data_year']).eq(3)).sum()
        print("Matched CMS rows:", f"{matched.sum():,}")
        print("Incorrect three-year CMS mappings:", f"{mismatched:,}")
        display(frame.loc[matched].groupby(['cms_data_year','prediction_year']).size().rename('matched_rows'))
    else:
        print('CMS source-year column not available in this export; cannot validate row-level timing.')
    return {'rows':len(frame), 'positives':int(frame['shortage_target'].sum()),
            'duplicates':int(frame.duplicated(['package_ndc','prediction_year']).sum())}

strict_v04_result = audit_panel(strict_v04_panel, 'STRICT v0.4 PANEL')
strict_v05_result = audit_panel(strict_panel, 'STRICT v0.5 PANEL')
print('\nReference only — legacy panel exists:', legacy_panel.is_file())

## 2. Check the historical FDA label definition

The source snapshot's `status` field distinguishes **Current**, **Resolved**, and **To Be Discontinued** records. All three can be linked by the current target-generation rule, but they do not represent identical business events. This cell audits the status composition of *source records linked to positive labels*, without changing those labels.

In [ ]:
if not (source_zip.is_file() and legacy_panel.is_file()):
    print("Source ZIP or legacy panel unavailable; skipping label-status audit.")
else:
    with ZipFile(source_zip) as z:
        members = [x for x in z.namelist() if x.lower().endswith('.json')]
        if not members:
            raise ValueError('No JSON member in FDA shortage ZIP')
        with z.open(members[0]) as f:
            raw = json.load(f)
    records = raw['results'] if isinstance(raw, dict) else raw
    source = pd.json_normalize(records)
    for c in ['package_ndc','initial_posting_date','status']:
        if c not in source:
            raise ValueError(f'FDA source missing {c}')
    source['package_ndc'] = source['package_ndc'].astype('string').str.strip()
    source['prediction_year'] = pd.to_datetime(
        source['initial_posting_date'], errors='coerce', format='mixed'
    ).dt.year
    source = source.loc[source['prediction_year'].isin([2022,2023,2024,2025])]
    labels = pd.read_csv(legacy_panel,
        usecols=['package_ndc','prediction_year','shortage_target'],
        dtype={'package_ndc':'string'}, low_memory=False)
    positives = labels.loc[labels['shortage_target'].eq(1)].copy()
    positives['package_ndc'] = positives['package_ndc'].str.strip()
    linked = positives.merge(source[['package_ndc','prediction_year','status']],
        on=['package_ndc','prediction_year'], how='left', validate='one_to_many', indicator=True)
    print('Positive package-year labels:', len(positives))
    print('Positive package-years with no matching source:',
          linked.loc[linked['_merge'].eq('left_only'), ['package_ndc','prediction_year']].drop_duplicates().shape[0])
    status_counts = linked.groupby(['prediction_year','status'],dropna=False)['package_ndc'].nunique().rename('positive_package_ndcs')
    display(status_counts.to_frame())
    print('\nIMPORTANT: Status is taken from the current retrospective FDA snapshot,')
    print('not necessarily the status on the historical prediction date.')
    print('Discontinuation notices should be separately considered when defining the outcome.')

## 3. Keep old model metrics separate from the corrected experiment

The old Notebook 12 reported **10** positive labels in the top 500 for v0.4 and **6** for v0.5 on the previously inspected 2025 cohort. Those are **legacy results**; do not relabel them as strict-CMS results. This cell checks whether new strict-CMS score files exist, but does not calculate metrics from unverified score definitions.

In [ ]:
print('LEGACY SCORES:', 'found' if legacy_scores.is_file() else 'not found')
print('STRICT-CMS SCORES:', 'found' if strict_scores.is_file() else 'not generated yet')
if strict_scores.is_file():
    cols = pd.read_csv(strict_scores, nrows=0).columns.tolist()
    print('Strict score columns:', cols)
    print('Next step: verify the score cohort, targets and ranking method before comparison.')
else:
    print('No corrected model comparison is available yet. That is expected if Notebook 11')
    print('was stopped after the CMS merge. Do NOT treat the legacy scores as corrected scores.')
print('\nAUDIT FINISHED. No files modified, no models retrained.')

## Interpretation and next decision

- **If strict v0.4 and v0.5 panels are missing:** Notebook 11's merge may have passed without its export cells being run. Do not run the training section automatically; inspect the export stage first.
- **If strict panels exist and their grain and CMS lag checks pass:** The *integration* is ready, but label validity remains a separate research decision.
- **If the 2025 target includes many “To Be Discontinued” records:** Decide whether PharmaOpt's outcome is *any FDA shortage-feed event*, *active shortage only*, or a multi-class outcome. Preserve existing experiments and document the definition before retraining.
- **Do not claim model performance has improved** until a corrected, appropriately evaluated model has been trained. The 2025 cohort is a development holdout, not an untouched test.
